In [ ]:
!apt-get update -qq && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

^C
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%pip -q install langchain-core langchain-ollama pydantic


In [ ]:
# Переключатели режима. Для быстрой проверки оставьте USE_LLM = False.
USE_LLM = True
USE_GOOGLE_DRIVE = True  # True в Colab, если файл памяти нужно хранить на Google Drive
MODEL_NAME = "llama3.2:1b"  # компактнее исходной llama3.2, быстрее на CPU
OLLAMA_TIMEOUT = 90

import subprocess
import shutil
import time
import urllib.request

if USE_LLM and shutil.which("ollama") is None:
    print("Устанавливаем Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
elif USE_LLM:
    print("Ollama уже установлена.")
else:
    print("Быстрый режим: установка Ollama пропущена.")


Ollama уже установлена.


In [ ]:
OLLAMA_URL = "http://127.0.0.1:11434"
ollama_process = None

if not USE_LLM:
    print("Быстрый режим: запуск сервера Ollama не требуется.")
else:
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama не найдена. Сначала выполните предыдущую ячейку.")
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)
        print("Ollama уже запущена.")
    except Exception:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        for attempt in range(20):
            try:
                urllib.request.urlopen(OLLAMA_URL, timeout=1)
                print("Ollama запущена.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("Не удалось запустить Ollama за 20 секунд. Проверьте её установку.")


Ollama запущена.


In [ ]:
if USE_LLM:
    # При первом запуске скачивает модель; при следующих запусках Ollama использует локальную копию.
    subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
else:
    print("Быстрый режим: скачивание модели пропущено.")


In [ ]:
from pathlib import Path
import json
import os

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/langchain_memory_colab")
else:
    # Для локального запуска и быстрого теста — папка рядом с notebook.
    PROJECT_DIR = Path.cwd()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_FILE = PROJECT_DIR / "memory.json"
USER_ID = "student_1"
BUFFER_MESSAGES = 6  # 3 последние пары «пользователь — агент»

print("Папка проекта:", PROJECT_DIR)
print("Файл памяти:", MEMORY_FILE)
print("Режим:", "LLM" if USE_LLM else "FAST DEMO")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Папка проекта: /content/drive/MyDrive/langchain_memory_colab
Файл памяти: /content/drive/MyDrive/langchain_memory_colab/memory.json
Режим: LLM


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Literal
import re

llm = None
if USE_LLM:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(
        model=MODEL_NAME,
        temperature=0.1,
        num_predict=192,
        keep_alive="10m",
        client_kwargs={"timeout": OLLAMA_TIMEOUT},
    )
    print("Модель LangChain подключена:", MODEL_NAME)
else:
    print("LLM не запускается: быстрый режим использует локальный анализ для демонстрации памяти.")


Модель LangChain подключена: llama3.2:1b


In [ ]:
LEVELS = ["beginner", "intermediate", "advanced"]


def normalize_level(value: str | None) -> str | None:
    """Преобразует свободное описание уровня к одной из трёх ступеней."""
    if not value:
        return None
    text = value.lower().strip()
    if any(word in text for word in ["начал", "нович", "beginner", "basic", "a1", "a2"]):
        return "beginner"
    if any(word in text for word in ["средн", "intermediate", "middle", "b1", "b2"]):
        return "intermediate"
    if any(word in text for word in ["продвин", "эксперт", "advanced", "специалист", "c1", "c2"]):
        return "advanced"
    return None


def empty_user_record():
    return {
        "profile": {
            "name": None,
            "skills": [],
            "declared_level": None,
            "goals": [],
        },
        "learning_state": {
            "current_level": "beginner",
            "level_source": "default",
            "explanation_style": "step_by_step",
            "interaction_tone": "friendly",
            "negative_feedback_count": 0,
            "high_insistence_count": 0,
        },
        "summary": "",
        "recent_messages": [],
        "pending_summary": [],
    }


class FileMemoryStore:
    """Читает и атомарно сохраняет JSON-память с отдельными профилями пользователей."""

    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.data = self._read_file()

    def _read_file(self):
        if not self.path.exists():
            return {}
        try:
            with self.path.open("r", encoding="utf-8") as file:
                data = json.load(file)
            return data if isinstance(data, dict) else {}
        except (json.JSONDecodeError, OSError):
            # Не затираем исходный повреждённый файл молча.
            raise RuntimeError(f"Не удалось прочитать файл памяти: {self.path}")

    def save(self):
        temp_path = self.path.with_suffix(self.path.suffix + ".tmp")
        with temp_path.open("w", encoding="utf-8") as file:
            json.dump(self.data, file, ensure_ascii=False, indent=2)
        temp_path.replace(self.path)

    def ensure_user(self, user_id: str):
        if user_id not in self.data:
            self.data[user_id] = empty_user_record()
            self.save()
        else:
            # Добавляем новые поля при открытии старой версии файла.
            defaults = empty_user_record()
            for key, value in defaults.items():
                self.data[user_id].setdefault(key, value)
            for section in ("profile", "learning_state"):
                for key, value in defaults[section].items():
                    self.data[user_id][section].setdefault(key, value)
            self.save()

    def get_record(self, user_id: str):
        self.ensure_user(user_id)
        return self.data[user_id]

    def update_profile(self, user_id: str, facts: dict):
        record = self.get_record(user_id)
        profile = record["profile"]
        if facts.get("name"):
            profile["name"] = facts["name"].strip()
        if facts.get("level"):
            profile["declared_level"] = facts["level"].strip()
            normalized = normalize_level(facts["level"])
            if normalized:
                record["learning_state"]["current_level"] = normalized
                record["learning_state"]["level_source"] = "self_reported"
        for field in ("skills", "goals"):
            for item in facts.get(field, []) or []:
                item = str(item).strip()
                if item and item not in profile[field]:
                    profile[field].append(item)
        self.save()
        return profile

    def get_recent_messages(self, user_id: str):
        return list(self.get_record(user_id)["recent_messages"])

    def set_history(self, user_id: str, recent_messages: list, pending_summary: list):
        record = self.get_record(user_id)
        record["recent_messages"] = recent_messages
        record["pending_summary"] = pending_summary
        self.save()

    def set_summary(self, user_id: str, summary: str):
        self.get_record(user_id)["summary"] = summary
        self.save()

    def clear_pending_summary(self, user_id: str):
        self.get_record(user_id)["pending_summary"] = []
        self.save()


STORE = FileMemoryStore(MEMORY_FILE)
STORE.ensure_user(USER_ID)
print("Память загружена. Пользователь:", USER_ID)

Память загружена. Пользователь: student_1


In [ ]:
class UserFacts(BaseModel):
    name: str | None = Field(default=None, description="Имя пользователя, если он сам его назвал")
    skills: list[str] = Field(default_factory=list, description="Навыки, явно названные пользователем")
    level: str | None = Field(default=None, description="Самооценка уровня знаний пользователя")
    goals: list[str] = Field(default_factory=list, description="Личные учебные или карьерные цели пользователя")


class IncomingAnalysis(BaseModel):
    question_class: Literal[
        "greeting", "conceptual_question", "code_help", "practical_task",
        "career_advice", "profile_update", "feedback", "other"
    ] = "other"
    sentiment: Literal["positive", "neutral", "negative"] = "neutral"
    insistence: Literal["low", "medium", "high"] = "low"
    difficulty_signal: Literal["too_easy", "too_hard", "appropriate", "unknown"] = "unknown"
    topic: str | None = None
    facts: UserFacts = Field(default_factory=UserFacts)


analysis_chain = None
if USE_LLM:
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", """
Ты классификатор входящих сообщений учебного чат-агента. Верни только данные по JSON-схеме.
Классы: greeting, conceptual_question, code_help, practical_task, career_advice,
profile_update, feedback, other. sentiment относится к реакции пользователя на диалог.
insistence=high ставь только при явной настойчивости или срочности.
difficulty_signal=too_hard при просьбе упростить, too_easy при просьбе усложнить.
Извлекай только явно сообщённые факты о самом пользователе. Не додумывай данные.
"""),
        ("human", "Сообщение пользователя: {message}"),
    ])
    analysis_chain = analysis_prompt | llm.with_structured_output(
        IncomingAnalysis, method="json_mode"
    )


def fast_analyze_message(message: str) -> IncomingAnalysis:
    """Быстрый локальный разбор для проверки логики без ожидания LLM."""
    text = message.lower().strip()
    result = IncomingAnalysis()

    # Класс сообщения
    if text.startswith(("меня зовут", "я изучаю", "хочу стать", "мой уровень", "моя цель")):
        result.question_class = "profile_update"
    elif any(x in text for x in ["сложно", "проще", "по шагам", "не понимаю", "слишком легко"]):
        result.question_class = "feedback"
    elif any(x in text for x in ["привет", "здравствуй", "добрый день"]):
        result.question_class = "greeting"
    elif any(x in text for x in ["ошибка", "код", "python", "sql", "не работает"]):
        result.question_class = "code_help"
    elif any(x in text for x in ["что такое", "объясни", "расскажи"]):
        result.question_class = "conceptual_question"

    # Извлечение фактов в примере демонстрации
    name_match = re.search(r"меня зовут\s+([а-яёa-z-]+)", text, flags=re.IGNORECASE)
    if name_match:
        result.facts.name = name_match.group(1).capitalize()
    if "python" in text:
        result.facts.skills = ["Python"]
    if "начальн" in text or "нович" in text:
        result.facts.level = "начальный"
    elif "средн" in text:
        result.facts.level = "средний"
    elif "продвин" in text:
        result.facts.level = "продвинутый"
    if "хочу стать аналитиком данных" in text:
        result.facts.goals = ["стать аналитиком данных"]
    elif "хочу стать" in text:
        goal = text.split("хочу стать", 1)[1].strip(" .,!?")
        if goal:
            result.facts.goals = ["стать " + goal]

    # Сигналы адаптации
    if any(x in text for x in ["сложно", "проще", "не понимаю", "по шагам", "объясни понятнее"]):
        result.difficulty_signal = "too_hard"
    elif any(x in text for x in ["слишком легко", "можно сложнее", "повысить сложность"]):
        result.difficulty_signal = "too_easy"
    if any(x in text for x in ["плохо объяснил", "не то", "неправильно", "бесполезно"]):
        result.sentiment = "negative"
    if any(x in text for x in ["повторяю", "я уже говорил", "срочно", "давай быстрее"]):
        result.insistence = "high"
    return result

print("Классификатор и структура извлекаемых фактов готовы.")


Классификатор и структура извлекаемых фактов готовы.


In [ ]:
class SaveFactsInput(BaseModel):
    user_id: str
    name: str | None = None
    skills: list[str] = Field(default_factory=list)
    level: str | None = None
    goals: list[str] = Field(default_factory=list)


@tool("save_user_facts", args_schema=SaveFactsInput)
def save_user_facts(user_id: str, name: str | None = None,
                    skills: list[str] | None = None, level: str | None = None,
                    goals: list[str] | None = None) -> str:
    """Сохраняет подтверждённые факты о пользователе в JSON-файл."""
    profile = STORE.update_profile(user_id, {
        "name": name,
        "skills": skills or [],
        "level": level,
        "goals": goals or [],
    })
    return "Факты сохранены в memory.json."


class LearningUpdateInput(BaseModel):
    user_id: str
    difficulty_signal: Literal["too_easy", "too_hard", "appropriate", "unknown"] = "unknown"
    sentiment: Literal["positive", "neutral", "negative"] = "neutral"
    insistence: Literal["low", "medium", "high"] = "low"


@tool("update_learning_state", args_schema=LearningUpdateInput)
def update_learning_state(user_id: str, difficulty_signal: str = "unknown",
                          sentiment: str = "neutral", insistence: str = "low") -> str:
    """Адаптирует сложность и стиль объяснения по обратной связи пользователя."""
    record = STORE.get_record(user_id)
    state = record["learning_state"]
    current = state.get("current_level", "beginner")
    if current not in LEVELS:
        current = "beginner"

    if difficulty_signal == "too_hard":
        state["current_level"] = LEVELS[max(0, LEVELS.index(current) - 1)]
        state["explanation_style"] = "step_by_step"
    elif difficulty_signal == "too_easy":
        state["current_level"] = LEVELS[min(len(LEVELS) - 1, LEVELS.index(current) + 1)]
        state["explanation_style"] = "more_advanced"
    elif difficulty_signal == "appropriate":
        state["explanation_style"] = "balanced"

    # Негативная реакция меняет тон, но сама по себе не снижает знания пользователя.
    if sentiment == "negative":
        state["negative_feedback_count"] += 1
        state["interaction_tone"] = "calm_supportive"
    if insistence == "high":
        state["high_insistence_count"] += 1
        state["interaction_tone"] = "direct_and_concise"

    STORE.save()
    return (
        f"Настройки обновлены: уровень сложности={state['current_level']}; "
        f"стиль={state['explanation_style']}; тон={state['interaction_tone']}."
    )

print("Инструменты сохранения фактов и адаптации созданы.")

Инструменты сохранения фактов и адаптации созданы.


In [ ]:
class FileChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, user_id: str, store: FileMemoryStore):
        self.user_id = user_id
        self.store = store

    @property
    def messages(self) -> list[BaseMessage]:
        result = []
        for item in self.store.get_recent_messages(self.user_id):
            if item.get("role") == "user":
                result.append(HumanMessage(content=item.get("content", "")))
            elif item.get("role") == "assistant":
                result.append(AIMessage(content=item.get("content", "")))
        return result

    def add_messages(self, messages: list[BaseMessage]) -> None:
        record = self.store.get_record(self.user_id)
        recent = list(record["recent_messages"])
        pending = list(record["pending_summary"])

        for message in messages:
            if isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                continue
            content = message.content
            if not isinstance(content, str):
                content = json.dumps(content, ensure_ascii=False, default=str)
            recent.append({"role": role, "content": content})

        while len(recent) > BUFFER_MESSAGES:
            pending.append(recent.pop(0))

        self.store.set_history(self.user_id, recent, pending)

    def clear(self) -> None:
        self.store.set_history(self.user_id, [], [])
        self.store.set_summary(self.user_id, "")


_history_objects = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Возвращает отдельную файловую историю для каждого session_id."""
    if session_id not in _history_objects:
        _history_objects[session_id] = FileChatMessageHistory(session_id, STORE)
    return _history_objects[session_id]

print("Файловый buffer готов.")

Файловый buffer готов.


In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты создаёшь краткое резюме учебного диалога для долгосрочного контекста.
Сохраняй темы, уже разобранные понятия, текущую задачу и предпочтения в объяснениях.
Не дублируй профиль пользователя и не добавляй предположений. Пиши компактно на русском.
"""),
    ("human", "Предыдущее резюме:\n{old_summary}\n\nНовые сообщения:\n{messages}\n\nОбновлённое резюме:"),
])
summary_chain = (summary_prompt | llm) if USE_LLM else None


def summarize_pending_messages(user_id: str) -> bool:
    record = STORE.get_record(user_id)
    pending = record.get("pending_summary", [])
    if not pending:
        return False

    formatted = "\n".join(
        f"{'Пользователь' if item['role'] == 'user' else 'Агент'}: {item['content']}"
        for item in pending
    )
    if USE_LLM:
        response = summary_chain.invoke({
            "old_summary": record.get("summary", ""),
            "messages": formatted,
        })
        new_summary = response.content if hasattr(response, "content") else str(response)
    else:
        # Быстрое extractive-резюме: сохраняем содержание старых реплик без вызова модели.
        previous = record.get("summary", "").strip()
        new_summary = (previous + " " + formatted).strip()
        new_summary = new_summary[-900:]

    STORE.set_summary(user_id, new_summary.strip())
    STORE.clear_pending_summary(user_id)
    return True

print("Суммаризация готова: LLM-режим или быстрый extractive fallback.")


Суммаризация готова: LLM-режим или быстрый extractive fallback.


In [ ]:
chat_chain = None
if USE_LLM:
    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", "{system_context}"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ])
    base_chain = chat_prompt | llm
    chat_chain = RunnableWithMessageHistory(
        base_chain,
        get_session_history,
        input_messages_key="question",
        history_messages_key="history",
    )

QUESTION_INSTRUCTIONS = {
    "greeting": "Ответь дружелюбно и коротко, без лишней теории.",
    "conceptual_question": "Объясни понятие на уровне пользователя, дай простой пример и короткий итог.",
    "code_help": "Покажи рабочий код, затем объясни ключевые части и предложи способ проверить результат.",
    "practical_task": "Разбей решение на последовательные шаги, отметь важные проверки и типичные ошибки.",
    "career_advice": "Свяжи рекомендации с целями и навыками пользователя. Не приписывай ему неизвестный опыт.",
    "profile_update": "Кратко подтверди, какие сведения обновлены, и продолжи диалог.",
    "feedback": "Учти обратную связь. Не спорь с пользователем и скорректируй подачу.",
    "other": "Ответь по существу, учитывая сохранённый профиль и контекст.",
}

LAST_DIAGNOSTICS = {}


def analyze_message(message: str) -> IncomingAnalysis:
    if USE_LLM:
        try:
            return analysis_chain.invoke({"message": message})
        except Exception as error:
            LAST_DIAGNOSTICS["analysis_error"] = str(error)
            return IncomingAnalysis()
    return fast_analyze_message(message)


def parse_incoming_parameters(payload: dict) -> dict:
    """Кастомный компонент: разбирает вход, вызывает tools и обновляет JSON-память."""
    message = str(payload["message"])
    user_id = str(payload.get("user_id", USER_ID))
    analysis = analyze_message(message)
    actions = []

    facts = analysis.facts.model_dump()
    has_facts = any([facts.get("name"), facts.get("level"), facts.get("skills"), facts.get("goals")])
    if has_facts:
        actions.append(str(save_user_facts.invoke({"user_id": user_id, **facts})))

    needs_learning_update = (
        analysis.difficulty_signal != "unknown"
        or analysis.sentiment == "negative"
        or analysis.insistence == "high"
    )
    if needs_learning_update:
        actions.append(str(update_learning_state.invoke({
            "user_id": user_id,
            "difficulty_signal": analysis.difficulty_signal,
            "sentiment": analysis.sentiment,
            "insistence": analysis.insistence,
        })))
    return {"analysis": analysis, "actions": actions, "user_id": user_id}


input_controller = RunnableLambda(parse_incoming_parameters)


def make_system_context(user_id: str, analysis: IncomingAnalysis) -> str:
    record = STORE.get_record(user_id)
    profile = record["profile"]
    state = record["learning_state"]
    level_text = {
        "beginner": "начальный — без лишнего жаргона, простые примеры",
        "intermediate": "средний — профильные термины с пояснениями",
        "advanced": "продвинутый — технические детали и альтернативы",
    }.get(state.get("current_level"), "начальный — объясняй доступно")
    tone_text = {
        "friendly": "доброжелательный и спокойный",
        "calm_supportive": "спокойный и поддерживающий",
        "direct_and_concise": "прямой и лаконичный",
    }.get(state.get("interaction_tone"), "доброжелательный и спокойный")
    instruction = QUESTION_INSTRUCTIONS.get(analysis.question_class, QUESTION_INSTRUCTIONS["other"])
    return f"""Ты — учебный чат-агент на русском языке.
Профиль: имя={profile.get('name') or 'не указано'}; навыки={', '.join(profile.get('skills', [])) or 'не указаны'};
уровень={profile.get('declared_level') or 'не указан'}; цели={', '.join(profile.get('goals', [])) or 'не указаны'}.
Сложность: {state.get('current_level')} ({level_text}). Тон: {tone_text}.
Резюме диалога: {record.get('summary') or 'пока нет'}.
Формат ответа для класса {analysis.question_class}: {instruction}
Не выдумывай сведения о пользователе и не раскрывай внутренние tools или JSON."""


def fast_demo_response(message: str, analysis: IncomingAnalysis, user_id: str) -> str:
    """Шаблонные ответы только для демонстрации памяти и логики без долгой генерации."""
    record = STORE.get_record(user_id)
    profile = record["profile"]
    state = record["learning_state"]
    if any(x in message.lower() for x in ["как меня зовут", "какая у меня цель", "что я изучаю"]):
        return (f"Тебя зовут {profile.get('name') or 'имя пока не сохранено'}. "
                f"Ты изучаешь {', '.join(profile.get('skills', [])) or 'навыки пока не сохранены'}. "
                f"Твоя цель — {', '.join(profile.get('goals', [])) or 'цель пока не сохранена'}.")
    if analysis.question_class == "profile_update":
        name = profile.get("name") or "пользователь"
        return (f"Приятно познакомиться, {name}! Я сохранил твои сведения: "
                f"навыки — {', '.join(profile.get('skills', [])) or 'не указаны'}, "
                f"цель — {', '.join(profile.get('goals', [])) or 'не указана'}. ")
    if analysis.question_class == "feedback":
        return ("Понял. Буду объяснять проще и по шагам, с небольшими примерами. "
                f"Текущий уровень сложности: {state.get('current_level')}.")
    if analysis.question_class == "greeting":
        return "Привет! Я готов помочь с учёбой."
    return (f"Запрос принят. Я учитываю твой уровень ({state.get('current_level')}) "
            "и сохранённый контекст. В быстром режиме это демонстрационный ответ-шаблон.")


fast_chat_chain = None
if not USE_LLM:
    fast_base_chain = RunnableLambda(
        lambda inputs: fast_demo_response(inputs["question"], inputs["analysis"], inputs["user_id"])
    )
    fast_chat_chain = RunnableWithMessageHistory(
        fast_base_chain,
        get_session_history,
        input_messages_key="question",
        history_messages_key="history",
    )


def chat(user_message: str, user_id: str = USER_ID) -> str:
    """Полный цикл: анализ -> tools -> ответ -> сохранение buffer -> summary."""
    LAST_DIAGNOSTICS.clear()
    prepared = input_controller.invoke({"message": user_message, "user_id": user_id})
    analysis = prepared["analysis"]
    LAST_DIAGNOSTICS.update({
        "analysis_mode": "LLM structured output" if USE_LLM else "local fast demo",
        "question_class": analysis.question_class,
        "sentiment": analysis.sentiment,
        "insistence": analysis.insistence,
        "difficulty_signal": analysis.difficulty_signal,
        "tool_actions": prepared["actions"],
    })

    if USE_LLM:
        response = chat_chain.invoke(
            {"system_context": make_system_context(user_id, analysis), "question": user_message},
            config={"configurable": {"session_id": user_id}},
        )
        answer = response.content if hasattr(response, "content") else str(response)
    else:
        answer = fast_chat_chain.invoke(
            {"question": user_message, "analysis": analysis, "user_id": user_id},
            config={"configurable": {"session_id": user_id}},
        )

    try:
        LAST_DIAGNOSTICS["summary_updated"] = summarize_pending_messages(user_id)
    except Exception as error:
        LAST_DIAGNOSTICS["summary_error"] = str(error)
        LAST_DIAGNOSTICS["summary_updated"] = False

    record = STORE.get_record(user_id)
    LAST_DIAGNOSTICS["current_level"] = record["learning_state"]["current_level"]
    LAST_DIAGNOSTICS["buffer_messages"] = len(record["recent_messages"])
    LAST_DIAGNOSTICS["pending_summary_messages"] = len(record["pending_summary"])
    LAST_DIAGNOSTICS["response_mode"] = "LLM" if USE_LLM else "шаблон для быстрой демонстрации"
    return answer

print("Основная цепочка и кастомный компонент input_controller готовы.")


Основная цепочка и кастомный компонент input_controller готовы.


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
def run_chat(user_id: str = USER_ID):
    print("Чат запущен. Для выхода введите: выход")
    print("Режим:", "LLM" if USE_LLM else "FAST DEMO (ответы-шаблоны, память работает реально)")
    while True:
        user_message = input("Ты: ").strip()
        if user_message.lower() in {"выход", "exit", "quit"}:
            print("Диалог завершён. Память сохранена в", MEMORY_FILE)
            break
        if not user_message:
            continue
        print("Агент:", chat(user_message, user_id=user_id))

# Для интерактивного режима раскомментируйте: run_chat()


In [ ]:
# Перед повторной демонстрацией очищаем только историю, но не профиль пользователя.
STORE.set_history(USER_ID, [], [])
STORE.set_summary(USER_ID, "")

# Шаг 1. Сообщаем сведения о пользователе и сохраняем их в memory.json.
first_message = "Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных. Мой уровень начальный."
print("Ты:", first_message)
print("Агент:", chat(first_message))


Ты: Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных. Мой уровень начальный.
Агент: Алексей,สวัสดь! Зналиste Python и хотите стать аналитиком данных? Это отличный выбор! В этой области много интересных задач и возможностей. Кто знает, может помочь?


In [ ]:
# Шаг 2. Проверяем, что агент обращается к сохранённому профилю.
question = "Как меня зовут, что я изучаю и какая у меня цель?"
print("Ты:", question)
print("Агент:", chat(question))


Ты: Как меня зовут, что я изучаю и какая у меня цель?
Агент: Алексей, я знаю, что вы изучаете Python и хотите стать аналитиком данных. Ваши цели могут быть разные, но я могу сказать, что в этой области часто используются навыки Python для работы с данными, анализирования данных и создания аналитических решений.


In [ ]:
# Шаг 3. Передаём сигнал, что объяснение слишком сложное.
feedback = "Мне сложно, объясни, пожалуйста, проще и по шагам."
print("Ты:", feedback)
print("Агент:", chat(feedback))


Ты: Мне сложно, объясни, пожалуйста, проще и по шагам.
Агент: Алексей, я объясняю, что аналитик данных — это человек, который analysирует данные, чтобы получить информацию и сделать выводы. В этой области часто используются навыки Python для работы с данными, анализирования данных и создания аналитических решений.


In [ ]:
# Шаг 4. Ещё одно сообщение переполняет buffer и запускает суммаризацию старых сообщений.
summary_question = "Объясни, что такое база данных, в двух предложениях."
print("Ты:", summary_question)
print("Агент:", chat(summary_question))
print("Резюме обновлено:", STORE.get_record(USER_ID).get("summary") or "пока пусто")


Ты: Объясни, что такое база данных, в двух предложениях.
Агент: Алексей, база данных — это система, в которой хранятся данные, которые можно использовать для анализа и обобщения. Базы данных могут быть в виде базы данных relational (например, MySQL, PostgreSQL) или NoSQL (например, MongoDB, Cassandra).

АLEXЕЙ, ВОЧЕРТИЯ.
Резюме обновлено: Новые сообщения:
Пользователь: Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных. Мой уровень начальный.
Агент: Алексей,สวัสดь! Зналиste Python и хотите стать аналитиком данных? Это отличный выбор! В этой области много интересных задач и возможностей. Кто знает, может помочь?

Обновлённое резюме:
Аналитик данных
Окончательный уровень: начальный
Область: Python
Задачи: анализ данных, отслеживание изменений, создание аналитическихรายงานов
Предмет: Python, аналитика данных, аналитический анализ, отслеживание изменений, аналитические модели, аналитические модели данных, аналитические модели данных Python


In [ ]:

record = STORE.get_record(USER_ID)
print(json.dumps(record, ensure_ascii=False, indent=2))


{
  "name": "Алексей",
  "skills": [
    "Python",
    "Аналитика данных"
  ],
  "level": null,
  "goals": [
    "Стать квалифицированным аналитиком данных"
  ],
  "profile": {
    "name": null,
    "skills": [],
    "declared_level": null,
    "goals": []
  },
  "learning_state": {
    "current_level": "beginner",
    "level_source": "default",
    "explanation_style": "step_by_step",
    "interaction_tone": "friendly",
    "negative_feedback_count": 0,
    "high_insistence_count": 0
  },
  "summary": "Новые сообщения:\nПользователь: Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных. Мой уровень начальный.\nАгент: Алексей,สวัสดь! Зналиste Python и хотите стать аналитиком данных? Это отличный выбор! В этой области много интересных задач и возможностей. Кто знает, может помочь?\n\nОбновлённое резюме:\nАналитик данных\nОкончательный уровень: начальный\nОбласть: Python\nЗадачи: анализ данных, отслеживание изменений, создание аналитическихรายงานов\nПредмет: Python, аналити

In [ ]:
# Имитация повторного открытия приложения: создаём новое хранилище из того же файла.
reloaded_store = FileMemoryStore(MEMORY_FILE)
reloaded_store.ensure_user(USER_ID)
reloaded_record = reloaded_store.get_record(USER_ID)
print("Профиль после повторной загрузки:")
print(json.dumps(reloaded_record["profile"], ensure_ascii=False, indent=2))
print("Уровень и стиль:", reloaded_record["learning_state"]["current_level"], reloaded_record["learning_state"]["explanation_style"])
print("Суммаризация:", reloaded_record.get("summary") or "пока пусто")
print("Сообщений в buffer:", len(reloaded_record.get("recent_messages", [])))
print("Файл существует:", MEMORY_FILE.exists())


Профиль после повторной загрузки:
{
  "name": null,
  "skills": [],
  "declared_level": null,
  "goals": []
}
Уровень и стиль: beginner step_by_step
Суммаризация: Новые сообщения:
Пользователь: Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных. Мой уровень начальный.
Агент: Алексей,สวัสดь! Зналиste Python и хотите стать аналитиком данных? Это отличный выбор! В этой области много интересных задач и возможностей. Кто знает, может помочь?

Обновлённое резюме:
Аналитик данных
Окончательный уровень: начальный
Область: Python
Задачи: анализ данных, отслеживание изменений, создание аналитическихรายงานов
Предмет: Python, аналитика данных, аналитический анализ, отслеживание изменений, аналитические модели, аналитические модели данных, аналитические модели данных Python
Сообщений в buffer: 6
Файл существует: True
